# SUMScope — compute-efficient long-document retrieval

**Research implementation and BRIGHT evaluation** by Reyam Albalihi.

SUMScope adapts Periscope’s reasoning-based reranking by replacing repeated query-dependent strided probes with one reusable, query-independent document summary probe. Its scoring is best local relevance plus cached summary relevance.

This notebook includes embedding retrieval, query-independent document summaries, six reranking variants, token/call accounting, and evaluation tables. It depends on the original Periscope repository for chunking, grid formation, relevance log-odds, and metrics. See the top-level README and CREDITS.md for full attribution.


## 1. Install

In [ ]:
# Check the environment. Install dependencies before launching this notebook.
import importlib.util, sys
required = ["torch", "transformers", "datasets", "vllm", "pandas", "numpy"]
missing = [x for x in required if importlib.util.find_spec(x) is None]
print("Python:", sys.version.split()[0])
print("Missing packages:", missing or "none")
if missing:
    raise RuntimeError("Set up a compatible GPU environment first; see README.md. Missing: " + ", ".join(missing))


## 2. Setup

In [ ]:
import os
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"   # run the vLLM engine in this process: most robust in a notebook
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import gc, json, subprocess, sys, time
import numpy as np
import pandas as pd

REPO = os.environ.get("PERISCOPE_PATH", os.path.abspath("external/Periscope"))
os.makedirs(os.path.dirname(REPO), exist_ok=True)
COMMIT = "60da00aa7f3399c9a63f43792b9369b4ed140f0e"     # the version of the authors' code this notebook was checked against
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/mohammad2012191/Periscope.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from periscope import core as C       # chunk, grid, log_odds: the paper's exact implementation
from periscope import metrics as M    # ndcg, mrr, recall, rank: the paper's exact metrics

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/periscope_hackathon"
except ImportError:                    # not on Colab: keep results next to the notebook
    BASE = os.path.abspath(os.environ.get("SUMSCOPE_OUTPUT_DIR", "outputs/periscope_hackathon"))
os.makedirs(BASE, exist_ok=True)
print("results in", BASE)

## 3. Config

In [ ]:
DOMAINS   = ["earth_science", "biology", "pony", "psychology"]
N_QUERIES = 50               # first 50 queries per domain in this experiment
TOPK           = 100                   # candidates re-ranked per query
CHUNK          = 100                   # tokens per chunk, as in the paper for retrieval
WINDOW         = 16384                 # the paper's retrieval window (vLLM max_model_len)
MODEL          = "Qwen/Qwen3-4B-Instruct-2507"
EMBED_MODEL    = "Qwen/Qwen3-Embedding-4B"
GPU_UTIL       = 0.85                  # share of GPU memory vLLM may take
SUM_PIECE      = 12000                 # a document longer than this is summarized in pieces, then the pieces' summaries
SUM_MAX_PIECES = 20                    # cap for very long documents: 20 evenly spaced pieces
GROUP_PROMPTS  = 4000                  # probe prompts per vLLM call (bigger = faster, until memory runs out)


def summary_budget(n_chunks):
    """Summary length in tokens: one chunk below a 4x4 grid (16 chunks), two below 6x6 (36), three above."""
    if n_chunks < 16:
        return CHUNK
    if n_chunks < 36:
        return 2 * CHUNK
    return 3 * CHUNK


def needs_summary(n_chunks):
    """1-2 chunks form a single row, which already is the whole document: no summary.
    For these documents the 'summary' probe is the row probe itself (score 2 x row, as Periscope on 1-chunk docs)."""
    return n_chunks > 2


RUN = os.path.join(BASE, "q%d" % N_QUERIES)    # per-run files; summaries and document embeddings are shared across runs
os.makedirs(RUN, exist_ok=True)

## 4. Data (BRIGHT long documents, loaded exactly as the authors' `run_retrieval.py`)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)      # the probe model's tokenizer, used for chunking as in the paper
long_docs = load_dataset("xlangai/BRIGHT", "long_documents")
examples_all = load_dataset("xlangai/BRIGHT", "examples")

DATA = {}
for dom in DOMAINS:
    corpus = {str(x["id"]): x["content"] for x in long_docs[dom]}
    ex = list(examples_all[dom])[:N_QUERIES or None]
    queries = {str(e["id"]): e["query"] for e in ex}
    golds = {str(e["id"]): {str(g) for g in e.get("gold_ids_long", e.get("gold_ids", [])) if str(g) in corpus} for e in ex}
    qids = [q for q in queries if golds[q]]     # the paper's metrics skip queries with no gold document; we skip them everywhere
    DATA[dom] = dict(corpus=corpus, queries=queries, golds=golds, qids=qids)
    print("%-10s %d documents, %d queries (%d with a gold document)" % (dom, len(corpus), len(queries), len(qids)))

## 5. Stage 1: embedder ranking (GPU, ~5 min per domain the first time, then cached)
The authors' `baselines.py --method embed`: documents truncated to 2,048 tokens, queries to 512, last-token pooling, L2 normalization.

In [5]:
INSTRUCTION = "Instruct: Given a query, retrieve relevant documents.\nQuery: "
_EMB = {}


def _embedder():
    import torch
    from transformers import AutoModel
    if not _EMB:
        _EMB["tok"] = AutoTokenizer.from_pretrained(EMBED_MODEL, padding_side="left")
        _EMB["model"] = AutoModel.from_pretrained(EMBED_MODEL, torch_dtype=torch.bfloat16).cuda().eval()
    return _EMB["tok"], _EMB["model"]


def encode(items, max_len, batch=8):
    import torch
    etok, model = _embedder()
    out = []
    with torch.no_grad():
        for i in range(0, len(items), batch):
            b = etok(items[i:i + batch], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to("cuda")
            h = model(**b).last_hidden_state
            out.append(torch.nn.functional.normalize(h[:, -1], p=2, dim=1).float().cpu())   # last token (left padding)
    return torch.cat(out)


EMB = {}                     # EMB[dom][qid][doc] = cosine similarity, whole corpus
for dom in DOMAINS:
    path = os.path.join(RUN, "embed_%s.json" % dom)
    if os.path.exists(path):
        EMB[dom] = json.load(open(path))
        continue
    import torch
    d = DATA[dom]
    doc_ids = sorted(d["corpus"])
    dpath = os.path.join(BASE, "docemb_%s.pt" % dom)            # document vectors do not depend on the queries
    if os.path.exists(dpath):
        D = torch.load(dpath)
    else:
        t0 = time.time()
        D = encode([d["corpus"][x] for x in doc_ids], 2048)
        torch.save(D, dpath)
        print("%s: %d documents embedded in %.0fs" % (dom, len(doc_ids), time.time() - t0))
    Q = encode([INSTRUCTION + d["queries"][q] for q in d["qids"]], 512)
    sims = (Q @ D.T).numpy()
    EMB[dom] = {q: {doc_ids[j]: float(sims[i, j]) for j in range(len(doc_ids))} for i, q in enumerate(d["qids"])}
    json.dump(EMB[dom], open(path, "w"))

if _EMB:                       # free the GPU for vLLM
    import torch
    del _EMB["model"], _EMB["tok"]
    _EMB.clear()
    gc.collect()
    torch.cuda.empty_cache()

# Full-corpus embedder metrics, and the paper's Table 5 values for the same protocol (meaningful at N_QUERIES = 50)
PAPER_EMBED = {"biology": {"NDCG@10": .565, "MRR": .535, "R@10": .830},
               "pony":    {"NDCG@10": .143, "MRR": .336, "R@10": .145}}
rows = []
for dom in DOMAINS:
    m = M.report(EMB[dom], DATA[dom]["golds"])
    ref = PAPER_EMBED.get(dom, {})
    rows.append({"domain": dom, "n": m["n"], **{k: round(m[k], 3) for k in ("NDCG@10", "MRR", "R@10")},
                 **{"paper " + k: v for k, v in ref.items()}})
print("Embedder on the full corpus (sanity check against the paper's Table 5):")
display(pd.DataFrame(rows).set_index("domain"))
if N_QUERIES == 50:
    for r in rows:
        if "paper NDCG@10" in r and abs(r["NDCG@10"] - r["paper NDCG@10"]) > 0.03:
            print("WARNING: %s NDCG@10 %.3f is far from the paper's %.3f: check the setup before spending GPU time."
                  % (r["domain"], r["NDCG@10"], r["paper NDCG@10"]))
else:
    print("(N_QUERIES = %d: the paper's numbers are over 50 queries, so only a rough comparison)" % N_QUERIES)

Embedder on the full corpus (sanity check against the paper's Table 5):


,n,NDCG@10,MRR,R@10,paper NDCG@10,paper MRR,paper R@10
domain,,,,,,,
earth_science,50,0.578,0.580,0.773,NaN,NaN,NaN
biology,50,0.562,0.532,0.830,0.565,0.535,0.830
pony,50,0.148,0.334,0.152,0.143,0.336,0.145
psychology,50,0.561,0.521,0.772,NaN,NaN,NaN


## 6. Candidate pools, recall ceiling, and chunking (CPU)

In [6]:
ORDER, POOL, DOCQ, CH = {}, {}, {}, {}
ceiling = []
for dom in DOMAINS:
    d = DATA[dom]
    ORDER[dom] = {q: M.rank(EMB[dom][q]) for q in d["qids"]}            # full embedder ranking, as the paper ranks
    POOL[dom] = {q: ORDER[dom][q][:TOPK] for q in d["qids"]}
    DOCQ[dom] = {}                                                       # document -> the queries whose pool holds it
    for q in d["qids"]:
        for doc in POOL[dom][q]:
            DOCQ[dom].setdefault(doc, []).append(q)
    CH[dom] = {doc: C.chunk(tok, d["corpus"][doc], CHUNK) or ["empty"] for doc in sorted(DOCQ[dom])}
    r100 = np.mean([M.recall(d["golds"][q], POOL[dom][q], TOPK) for q in d["qids"]])
    hit = np.mean([bool(d["golds"][q] & set(POOL[dom][q])) for q in d["qids"]])
    n_chunks = [len(c) for c in CH[dom].values()]
    ceiling.append({"domain": dom, "Recall@%d" % TOPK: round(r100, 3), "queries with a gold doc in the pool": round(hit, 3),
                    "pooled docs": len(CH[dom]), "(query, doc) pairs": sum(len(v) for v in DOCQ[dom].values()),
                    "median chunks/doc": int(np.median(n_chunks)), "max chunks/doc": max(n_chunks),
                    "docs needing a summary": sum(needs_summary(n) for n in n_chunks)})
CEILING = pd.DataFrame(ceiling).set_index("domain")
print("The ceiling: no re-ranker can recover a gold document outside the top %d." % TOPK)
display(CEILING)
CEILING.to_csv(os.path.join(RUN, "recall_ceiling.csv"))

The ceiling: no re-ranker can recover a gold document outside the top 100.


,Recall@100,queries with a gold doc in the pool,pooled docs,"(query, doc) pairs",median chunks/doc,max chunks/doc,docs needing a summary
domain,,,,,,,
earth_science,0.947,1.00,555,5000,47,8650,547
biology,0.980,0.98,511,5000,47,4202,508
pony,0.874,1.00,342,5000,6,401,233
psychology,0.933,0.94,480,5000,55,2079,474


## 7. Stage 2: load the probe model (vLLM) and check the readout

In [7]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL, max_model_len=WINDOW, gpu_memory_utilization=GPU_UTIL,
          enable_prefix_caching=True, dtype="bfloat16", seed=0)


def _probe_params():
    # The paper's readout: one token, top-20 log-probabilities, temperature 1, no nucleus or top-k truncation.
    for top_k in (-1, 0):               # vLLM versions differ on how "no top-k" is spelled
        try:
            return SamplingParams(max_tokens=1, logprobs=20, temperature=1.0, top_p=1.0, top_k=top_k, min_p=0.0)
        except Exception:
            pass
    raise RuntimeError("could not build the probe sampling parameters")


PROBE = _probe_params()


def chat(body):
    # The authors' chat(): one user turn, generation prompt, thinking disabled where the template has it.
    return tok.apply_chat_template([{"role": "user", "content": body}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)


def encode_prompts(prompts):
    return tok(prompts, add_special_tokens=False)["input_ids"]


def top20(token_ids):
    """One forward pass per prompt; the top-20 log-probabilities of the first answer token, {surface: logprob}.
    vLLM may also return the sampled token when it falls outside the top 20: it is dropped, as the server's top_logprobs does."""
    out = []
    outs = llm.generate([{"prompt_token_ids": ids} for ids in token_ids], PROBE, use_tqdm=False)
    for o in outs:
        top = {}
        for v in o.outputs[0].logprobs[0].values():
            if v.rank is not None and v.rank <= 20 and v.decoded_token is not None:
                top[v.decoded_token] = max(top.get(v.decoded_token, -1e9), float(v.logprob))
        out.append(top)
    return out


def yes_score(top):
    return C.log_odds(top, ["Yes"], "No")["Yes"]        # Eq. 1: log p(Yes) - log p(No), the authors' function


REL = "Document: %s\nQuery: %s\nIs this document relevant to the query? Answer Yes or No:"        # the paper's prompt
REL_SUM = ("Document summary: %s\nPassage: %s\nQuery: %s\n"
           "Is this document relevant to the query? Answer Yes or No:")                            # summary prepended


def p_rel(span, query):
    return chat(REL % (span, query))


def p_rel_sum(summary, span, query):
    return chat(REL_SUM % (summary, span, query))


# Startup check, as the authors do: the answer tokens must be in the top 20 and point the right way.
t_yes, t_no = top20(encode_prompts([p_rel("The sky is blue on a clear day.", "What colour is the sky?"),
                                    p_rel("The sky is blue on a clear day.", "How do I bake sourdough bread?")]))
lp, _ = C.collapse(t_yes, {"Yes", "No"})
assert "Yes" in lp and "No" in lp, "Yes/No not in the top 20 of a trivial probe: check the chat template. Got %s" % sorted(t_yes)
print("relevant pair: %+.2f   irrelevant pair: %+.2f" % (yes_score(t_yes), yes_score(t_no)))
assert yes_score(t_yes) > 0 > yes_score(t_no), "the readout points the wrong way: check the prompt and the template"

(MainProcess pid=3632) INFO 10-30 13:16:54 [api_utils.py:286] non-default args: {'dtype': 'bfloat16', 'max_model_len': 16384, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}
(MainProcess pid=3632) WARNING 10-30 13:16:54 [arg_utils.py:1945] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=3632) INFO 10-30 13:17:03 [model.py:733] Resolved architecture: Qwen3ForCausalLM
(MainProcess pid=3632) INFO 10-30 13:17:03 [model.py:2152] Using max model len 16384
(MainProcess pid=3632) INFO 10-30 13:17:03 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=16384.


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/3 [00:00<?, ?it/s]

(MainProcess pid=3632) INFO 10-30 13:17:04 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

... (5 outputs left out)

2026-10-30 13:18:24,318 - INFO - autotuner.py:1046 - flashinfer.jit: [Autotuner]: Autotuning process starts ...


(MainProcess pid=3632) INFO 10-30 13:18:24 [kernel_warmup.py:367] Running FlashInfer autotune with 16384 tokens and token buckets (1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 768, 1024, 1280, 1536, 1792, 2048, 2560, 3072, 3584, 4096, 8192, 16384).


2026-10-30 13:18:47,037 - INFO - autotuner.py:1069 - flashinfer.jit: [Autotuner]: Autotuning process ends
2026-10-30 13:18:48,414 - INFO - core.py:564 - flashinfer.jit: Building JIT module sampling; this can take several minutes on first use.
Capturing CUDA graphs (FULL): 100%|██████████| 51/51 [00:39<00:00,  1.28it/s]


(MainProcess pid=3632) INFO 10-30 13:20:17 [model_runner.py:1127] Graph capturing finished in 43 secs, took 0.21 GiB
(MainProcess pid=3632) INFO 10-30 13:20:17 [gpu_worker.py:971] CUDA graph pool memory: 0.21 GiB (actual), 0.45 GiB (estimated), difference: 0.24 GiB (115.0%).
(MainProcess pid=3632) INFO 10-30 13:20:17 [gpu_worker.py:1034] Free memory on device (94.43/94.97 GiB) on startup. Desired GPU memory utilization is (0.85, 80.73 GiB). Actual usage is 8.48 GiB for consumed memory (weights + non-torch), 1.62 GiB for peak activation, and 0.21 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=75449796506` (70.27 GiB) to fit into requested memory, or `--kv-cache-memory=90160606720` (83.97 GiB) to fully utilize gpu memory. Current kv cache memory in use is 70.62 GiB.
(MainProcess pid=3632) INFO 10-30 13:20:17 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(MainProcess pid=3632) INFO

## 8. Stage 2: summaries (offline, once per document, cached on Drive)
Query-independent. A document longer than `SUM_PIECE` tokens is summarized in pieces, then the pieces' summaries are summarized
(repeated until it fits). Documents with more than `SUM_MAX_PIECES` pieces are summarized from evenly spaced pieces.

In [8]:
SUM_PROMPT = ("Summarize what the following document is about in about {words} words. Cover its main topics, key entities "
              "and claims, so that a reader can tell which questions it could help answer. Write only the summary.\n\n"
              "Document:\n{text}")
PIECE_PROMPT = ("The following is part {i} of {n} of a longer document. Summarize what this part is about in about {words} "
                "words. Cover its main topics, key entities and claims. Write only the summary.\n\nPart:\n{text}")
REDUCE_PROMPT = ("The following are summaries of consecutive parts of one document. Combine them into one summary of the "
                 "whole document in about {words} words. Cover its main topics, key entities and claims, so that a reader "
                 "can tell which questions it could help answer. Write only the summary.\n\nPart summaries:\n{text}")


def summarize(dom, docs):
    """Summaries of `docs`, batched across documents, one round per level of the recursion."""
    corpus = DATA[dom]["corpus"]
    budget = {doc: summary_budget(len(CH[dom][doc])) for doc in docs}
    stats = {doc: dict(budget=budget[doc], calls=0, prompt_tokens=0, gen_tokens=0, rounds=0, sampled=False) for doc in docs}
    state = {doc: ("doc", corpus[doc]) for doc in docs}       # what is left to summarize: the document, or its part summaries
    done = {}
    while state:
        jobs = []                                              # (doc, piece index or -1 for a final summary, prompt body)
        for doc, (kind, text) in state.items():
            words = int(budget[doc] * 0.7)
            ids = tok.encode(text, add_special_tokens=False)
            if len(ids) <= SUM_PIECE:
                body = (SUM_PROMPT if kind == "doc" else REDUCE_PROMPT).format(words=words, text=text)
                jobs.append((doc, -1, body))
            else:
                pieces = [ids[i:i + SUM_PIECE] for i in range(0, len(ids), SUM_PIECE)]
                idx = list(range(len(pieces)))
                if len(pieces) > SUM_MAX_PIECES:
                    idx = sorted(set(np.linspace(0, len(pieces) - 1, SUM_MAX_PIECES).round().astype(int).tolist()))
                    stats[doc]["sampled"] = True
                for i in idx:
                    piece = tok.decode(pieces[i], skip_special_tokens=True)
                    jobs.append((doc, i, PIECE_PROMPT.format(i=i + 1, n=len(pieces), words=words, text=piece)))
            stats[doc]["rounds"] += 1
        prompts = encode_prompts([chat(body) for _, _, body in jobs])
        params = [SamplingParams(temperature=0.0, max_tokens=budget[doc]) for doc, _, _ in jobs]
        outs = llm.generate([{"prompt_token_ids": p} for p in prompts], params, use_tqdm=True)
        parts = {}
        for (doc, i, _), p, o in zip(jobs, prompts, outs):
            st = stats[doc]
            st["calls"] += 1
            st["prompt_tokens"] += len(p)
            st["gen_tokens"] += len(o.outputs[0].token_ids)
            text = o.outputs[0].text.strip()
            if i == -1:
                if not text:                                   # an empty generation: fall back to the document's opening
                    text = tok.decode(tok.encode(corpus[doc], add_special_tokens=False)[:budget[doc]],
                                      skip_special_tokens=True)
                done[doc] = text
            else:
                parts.setdefault(doc, []).append((i, text))
        state = {doc: ("parts", "\n\n".join(t for _, t in sorted(ps))) for doc, ps in parts.items()}
    return {doc: dict(summary=done[doc], **stats[doc]) for doc in docs}


SUMS = {}                    # SUMS[dom][doc] = {"summary": ..., cost fields}; only documents that need one
for dom in DOMAINS:
    path = os.path.join(BASE, "summaries_%s.json" % dom)      # shared by every run: summaries do not depend on the queries
    SUMS[dom] = json.load(open(path)) if os.path.exists(path) else {}
    todo = [doc for doc in sorted(CH[dom]) if needs_summary(len(CH[dom][doc])) and doc not in SUMS[dom]]
    t0 = time.time()
    for i in range(0, len(todo), 200):                        # save every 200 documents
        SUMS[dom].update(summarize(dom, todo[i:i + 200]))
        json.dump(SUMS[dom], open(path, "w"))
        print("%s: %d / %d documents summarized (%.0fs)" % (dom, min(i + 200, len(todo)), len(todo), time.time() - t0))
    print("%s: %d summaries ready" % (dom, sum(doc in SUMS[dom] for doc in CH[dom])))

for dom in DOMAINS:          # one example to eyeball
    doc = next((x for x in sorted(CH[dom]) if x in SUMS[dom]), None)
    if doc:
        print("\n[%s] %s (%d chunks):\n%s" % (dom, doc, len(CH[dom][doc]), SUMS[dom][doc]["summary"]))

earth_science: 547 summaries ready
biology: 508 summaries ready
pony: 233 summaries ready
psychology: 474 summaries ready

[earth_science] -999_celsius_degree/Absolute_zero.txt (49 chunks):
This document explains the concept of absolute zero—the lowest possible temperature in thermodynamics, defined as 0 K (−273.15 °C or −459.67 °F), where particles have minimal vibrational motion and quantum zero-point energy remains. It discusses that absolute zero cannot be reached due to thermodynamic limits, as cooling approaches the temperature of the cooling agent asymptotically. Key claims include that matter at absolute zero is in its ground state with minimal internal energy, and that entropy approaches zero for perfect crystals, as per the third law of thermodynamics. The document covers quantum phenomena like Bose–Einstein condensates, superconductivity, and superfluidity that emerge near absolute zero. It details historical developments, from Boyle and Amontons to Lord Kelvin, and modern a

## 9. Stage 3: probes (the long stage; resumable)
For every pooled document and every query whose pool holds it: the K row probes, the K column probes, the K rows with the summary
prepended, and one summary probe. Rows-only and Periscope share the same row probes. A probe is issued only if its prompt fits the
window (prompt ≤ window − 16 tokens, the authors' rule). Each probe stores `[score, prompt tokens]`.

In [9]:
def doc_items(dom, doc):
    queries = DATA[dom]["queries"]
    _, local, strided = C.grid(CH[dom][doc])                 # the authors' grid: K = ceil(sqrt(N)), empty spans dropped
    summ = SUMS[dom][doc]["summary"] if doc in SUMS[dom] else None
    qs = DOCQ[dom][doc]
    items = []
    # Span-major order: every query for the SAME span is adjacent, so vLLM's prefix cache prefills the span once and
    # reuses it for all the queries (the query comes last in the prompt). Query-major order evicts the cache on long documents.
    for _, s in local:
        items += [(doc, q, "L", p_rel(s, queries[q])) for q in qs]
    for _, s in strided:
        items += [(doc, q, "S", p_rel(s, queries[q])) for q in qs]
    if summ is not None:
        for _, s in local:
            items += [(doc, q, "LS", p_rel_sum(summ, s, queries[q])) for q in qs]
        items += [(doc, q, "G", p_rel(summ, queries[q])) for q in qs]
    return items


def score_docs(dom, docs):
    items = [it for doc in docs for it in doc_items(dom, doc)]
    ids = encode_prompts([p for *_, p in items])
    keep = [i for i, x in enumerate(ids) if len(x) <= WINDOW - 16]
    tops = top20([ids[i] for i in keep]) if keep else []
    out = {doc: {q: {"L": [], "S": [], "LS": [], "G": [], "summarized": doc in SUMS[dom]} for q in DOCQ[dom][doc]}
           for doc in docs}
    for i, top in zip(keep, tops):
        doc, q, kind, _ = items[i]
        out[doc][q][kind].append([yes_score(top), len(ids[i])])
    for doc in docs:
        if doc not in SUMS[dom]:
            # No summary (1-2 chunks): the single row is the whole document, so the prepend arm reads the row alone,
            # and the summary probe is that same row probe (no extra call).
            for e in out[doc].values():
                e["LS"] = [list(x) for x in e["L"]]
                e["G"] = [list(max(e["L"]))] if e["L"] else []
    return out, len(items) - len(keep)


RES = {}                     # RES[dom][doc][qid] = {"L": [[score, tokens], ...], "S": ..., "LS": ..., "G": ..., "summarized": bool}
for dom in DOMAINS:
    path = os.path.join(RUN, "probes_%s.json" % dom)
    RES[dom] = json.load(open(path)) if os.path.exists(path) else {}
    todo = [doc for doc in sorted(DOCQ[dom]) if doc not in RES[dom]]
    n_items = {doc: len(DOCQ[dom][doc]) * (2 * C.grid(CH[dom][doc])[0] + (C.grid(CH[dom][doc])[0] + 1) * (doc in SUMS[dom]))
               for doc in todo}                              # approximate prompts per document, to size the batches
    print("%s: %d documents to probe (%d already done), ~%d prompts" % (dom, len(todo), len(RES[dom]), sum(n_items.values())))
    t0, done_prompts, skipped, group = time.time(), 0, 0, []
    for j, doc in enumerate(todo):
        group.append(doc)
        if sum(n_items[x] for x in group) >= GROUP_PROMPTS or j == len(todo) - 1:
            out, sk = score_docs(dom, group)
            RES[dom].update(out)
            skipped += sk
            done_prompts += sum(n_items[x] for x in group)
            json.dump(RES[dom], open(path, "w"))
            el = time.time() - t0
            print("  %d / %d docs, ~%d prompts, %.0fs elapsed, ~%.0f prompts/s" % (j + 1, len(todo), done_prompts, el,
                                                                                done_prompts / max(el, 1e-9)), flush=True)
            group = []
    if skipped:
        print("  %d probe prompts did not fit the window and were skipped (the paper skips them the same way)" % skipped)

earth_science: 130 documents to probe (425 already done), ~43630 prompts
  13 / 130 docs, ~9452 prompts, 139s elapsed, ~68 prompts/s
  30 / 130 docs, ~13980 prompts, 178s elapsed, ~79 prompts/s
  33 / 130 docs, ~18250 prompts, 228s elapsed, ~80 prompts/s
  40 / 130 docs, ~22266 prompts, 256s elapsed, ~87 prompts/s
  53 / 130 docs, ~26318 prompts, 341s elapsed, ~77 prompts/s
  70 / 130 docs, ~30959 prompts, 383s elapsed, ~81 prompts/s
  91 / 130 docs, ~35008 prompts, 422s elapsed, ~83 prompts/s
  109 / 130 docs, ~39080 prompts, 459s elapsed, ~85 prompts/s
  129 / 130 docs, ~43611 prompts, 519s elapsed, ~84 prompts/s
  130 / 130 docs, ~43630 prompts, 520s elapsed, ~84 prompts/s
biology: 511 documents to probe (0 already done), ~132292 prompts
  11 / 511 docs, ~4110 prompts, 25s elapsed, ~161 prompts/s
  30 / 511 docs, ~8266 prompts, 54s elapsed, ~152 prompts/s
  47 / 511 docs, ~12495 prompts, 78s elapsed, ~160 prompts/s
  69 / 511 docs, ~16977 prompts, 106s elapsed, ~160 prompts/s
  93 /

## 10. Results

In [10]:
NEG = -1e9
ARMS = ["embed", "rows", "periscope", "rows+sum (prepend)", "rows+sum (vote)", "summary only"]


def _best(entries):
    return max(s for s, _ in entries) if entries else NEG


def arm_score(e, arm):
    if arm == "rows":
        return _best(e["L"])
    if arm == "periscope":                     # the paper: a missing family counts as -1e9, a document with no probe at all is last
        return _best(e["L"]) + _best(e["S"]) if (e["L"] or e["S"]) else 2 * NEG
    if arm == "rows+sum (prepend)":
        return _best(e["LS"])
    if arm == "rows+sum (vote)":
        return _best(e["L"]) + _best(e["G"]) if (e["L"] or e["G"]) else 2 * NEG
    if arm == "summary only":
        return _best(e["G"])
    raise ValueError(arm)


def arm_cost(e, arm):
    """(calls, prompt tokens) the arm spends on one (query, document) pair, online."""
    fams = {"rows": ["L"], "periscope": ["L", "S"], "rows+sum (prepend)": ["LS"], "summary only": ["G"],
            "rows+sum (vote)": ["L", "G"] if e["summarized"] else ["L"]}[arm]     # unsummarized: G is the row probe itself
    return sum(len(e[f]) for f in fams), sum(t for f in fams for _, t in e[f])


def ranking(dom, q, arm):
    pool = POOL[dom][q]
    if arm != "embed":
        sc = {doc: arm_score(RES[dom][doc][q], arm) for doc in pool}
        pool = sorted(pool, key=lambda doc: -sc[doc])          # stable: ties keep the embedder's order
    return pool + ORDER[dom][q][TOPK:]                         # beyond the pool, the embedder's order


PQ, COST = {}, {}            # per-query metrics and costs
for arm in ARMS:
    for dom in DOMAINS:
        for q in DATA[dom]["qids"]:
            g, r = DATA[dom]["golds"][q], ranking(dom, q, arm)
            PQ[arm, dom, q] = {"NDCG@10": M.ndcg(g, r, 10), "MRR": M.mrr(g, r), "R@1": M.recall(g, r, 1), "R@10": M.recall(g, r, 10)}
            if arm == "embed":
                COST[arm, dom, q] = (0, 0)
            else:
                c = [arm_cost(RES[dom][doc][q], arm) for doc in POOL[dom][q]]
                COST[arm, dom, q] = (sum(x for x, _ in c), sum(t for _, t in c))

METRICS = ["NDCG@10", "MRR", "R@1", "R@10"]
mean = lambda xs: float(np.mean(xs)) if len(xs) else float("nan")

# Table 1: accuracy, macro mean over domains (as the paper averages), plus NDCG@10 per domain
rows = []
for arm in ARMS:
    per_dom = {dom: {m: mean([PQ[arm, dom, q][m] for q in DATA[dom]["qids"]]) for m in METRICS} for dom in DOMAINS}
    row = {"arm": arm}
    row.update({"NDCG@10 " + dom: per_dom[dom]["NDCG@10"] for dom in DOMAINS})
    row.update({m + " (mean)": mean([per_dom[dom][m] for dom in DOMAINS]) for m in METRICS})
    rows.append(row)
T_ACC = pd.DataFrame(rows).set_index("arm").round(3)

# Table 2: online cost per query
rows = []
for arm in ARMS:
    calls = mean([COST[arm, dom, q][0] for dom in DOMAINS for q in DATA[dom]["qids"]])
    toks = mean([COST[arm, dom, q][1] for dom in DOMAINS for q in DATA[dom]["qids"]])
    rows.append({"arm": arm, "LLM calls / query": round(calls), "prompt tokens / query (k)": round(toks / 1e3, 1)})
T_COST = pd.DataFrame(rows).set_index("arm")
ref = T_COST.loc["periscope", "prompt tokens / query (k)"]
T_COST["tokens vs periscope"] = (T_COST["prompt tokens / query (k)"] / ref).round(2)

# Table 3: offline summary cost (paid once per document, shared by every query)
rows = []
for dom in DOMAINS:
    s = [SUMS[dom][doc] for doc in CH[dom] if doc in SUMS[dom]]
    rows.append({"domain": dom, "docs summarized": len(s), "docs without summary (1-2 chunks)": len(CH[dom]) - len(s),
                 "LLM calls": sum(x["calls"] for x in s), "prompt tokens (M)": round(sum(x["prompt_tokens"] for x in s) / 1e6, 2),
                 "generated tokens (k)": round(sum(x["gen_tokens"] for x in s) / 1e3, 1),
                 "docs summarized from sampled pieces": sum(x["sampled"] for x in s)})
T_SUM = pd.DataFrame(rows).set_index("domain")

# Table 4: paired comparisons on NDCG@10 over all queries, with a bootstrap 95% interval
def paired(a, b, n_boot=2000, seed=0):
    keys = [(dom, q) for dom in DOMAINS for q in DATA[dom]["qids"]]
    diff = np.array([PQ[a, dom, q]["NDCG@10"] - PQ[b, dom, q]["NDCG@10"] for dom, q in keys])
    boots = np.random.default_rng(seed).choice(diff, size=(n_boot, len(diff)), replace=True).mean(axis=1)
    return {"A": a, "B": b, "mean diff (A-B)": round(diff.mean(), 3),
            "95% CI": "[%.3f, %.3f]" % tuple(np.percentile(boots, [2.5, 97.5])),
            "A wins": int((diff > 1e-9).sum()), "ties": int((np.abs(diff) <= 1e-9).sum()), "B wins": int((diff < -1e-9).sum())}

PAIRS = [("rows+sum (vote)", "periscope"), ("rows+sum (vote)", "rows"), ("rows+sum (vote)", "embed"),
         ("rows+sum (prepend)", "periscope"), ("rows+sum (prepend)", "rows"), ("rows+sum (vote)", "rows+sum (prepend)"),
         ("periscope", "rows"), ("periscope", "embed"), ("summary only", "embed")]
T_PAIR = pd.DataFrame([paired(a, b) for a, b in PAIRS]).set_index(["A", "B"])

# Table 5: everything per domain
rows = []
for dom in DOMAINS:
    for arm in ARMS:
        r = {"domain": dom, "arm": arm}
        r.update({m: round(mean([PQ[arm, dom, q][m] for q in DATA[dom]["qids"]]), 3) for m in METRICS})
        r["n"] = len(DATA[dom]["qids"])
        rows.append(r)
T_DOM = pd.DataFrame(rows).set_index(["domain", "arm"])

print("Queries: %s. Re-ranking the embedder's top %d." % (", ".join("%s %d" % (d, len(DATA[d]["qids"])) for d in DOMAINS), TOPK))
print("\nTable 1. Accuracy (mean = macro average over domains)");                 display(T_ACC)
print("\nTable 2. Online cost per query (the LLM calls of the re-rank)");         display(T_COST)
print("\nTable 3. Offline summary cost (once per document)");                     display(T_SUM)
print("\nTable 4. Paired NDCG@10 comparisons over all queries");                  display(T_PAIR)
print("\nTable 5. Per domain");                                                   display(T_DOM)
print("\nRecall ceiling");                                                        display(CEILING)

for name, t in [("accuracy", T_ACC), ("cost", T_COST), ("summary_cost", T_SUM), ("paired", T_PAIR), ("per_domain", T_DOM)]:
    t.to_csv(os.path.join(RUN, "table_%s.csv" % name))
print("\nTables saved to", RUN)

Queries: earth_science 50, biology 50, pony 50, psychology 50. Re-ranking the embedder's top 100.

Table 1. Accuracy (mean = macro average over domains)


,NDCG@10 earth_science,NDCG@10 biology,NDCG@10 pony,NDCG@10 psychology,NDCG@10 (mean),MRR (mean),R@1 (mean),R@10 (mean)
arm,,,,,,,,
embed,0.578,0.562,0.148,0.561,0.462,0.492,0.212,0.632
rows,0.503,0.676,0.443,0.590,0.553,0.614,0.310,0.659
periscope,0.480,0.692,0.422,0.636,0.558,0.601,0.298,0.681
rows+sum (prepend),0.485,0.650,0.401,0.485,0.505,0.543,0.206,0.683
rows+sum (vote),0.533,0.677,0.404,0.641,0.564,0.647,0.322,0.672
summary only,0.510,0.666,0.358,0.624,0.540,0.614,0.295,0.662



Table 2. Online cost per query (the LLM calls of the re-rank)


,LLM calls / query,prompt tokens / query (k),tokens vs periscope
arm,,,
embed,0,0.0,0.00
rows,840,1472.6,0.50
periscope,1728,2952.0,1.00
rows+sum (prepend),840,1670.8,0.57
rows+sum (vote),935,1504.7,0.51
summary only,100,33.0,0.01



Table 3. Offline summary cost (once per document)


,docs summarized,docs without summary (1-2 chunks),LLM calls,prompt tokens (M),generated tokens (k),docs summarized from sampled pieces
domain,,,,,,
earth_science,547,8,1502,11.90,340.5,10
biology,508,3,801,4.66,185.3,1
pony,233,109,249,0.60,36.9,0
psychology,474,6,906,5.97,209.7,0


... (3 outputs left out)

NDCG@10    MRR    R@1   R@10   n
domain        arm                                                 
earth_science embed                 0.578  0.580  0.285  0.773  50
              rows                  0.503  0.522  0.307  0.637  50
              periscope             0.480  0.466  0.217  0.668  50
              rows+sum (prepend)    0.485  0.458  0.190  0.733  50
              rows+sum (vote)       0.533  0.532  0.300  0.732  50
              summary only          0.510  0.489  0.217  0.747  50
biology       embed                 0.562  0.532  0.228  0.830  50
              rows                  0.676  0.663  0.415  0.823  50
              periscope             0.692  0.680  0.408  0.833  50
              rows+sum (prepend)    0.650  0.638  0.370  0.823  50
              rows+sum (vote)       0.677  0.685  0.398  0.823  50
              summary only          0.666  0.677  0.418  0.807  50
pony          embed                 0.148  0.334  0.032  0.152  50
              rows                  0.443  0.705  0.098  0.434  50
              periscope             0.422  0.637  0.078  0.456  50
              rows+sum (prepend)    0.401  0.678  0.085  0.400  50
              rows+sum (vote)       0.404  0.747  0.111  0.354  50
              summary only          0.358  0.681  0.098  0.319  50
psychology    embed                 0.561  0.521  0.304  0.772  50
              rows                  0.590  0.568  0.420  0.741  50
              periscope             0.636  0.622  0.490  0.765  50
              rows+sum (prepend)    0.485  0.398  0.180  0.775  50
              rows+sum (vote)       0.641  0.625  0.480  0.781  50
              summary only          0.624  0.610  0.447  0.775  50


Recall ceiling


,Recall@100,queries with a gold doc in the pool,pooled docs,"(query, doc) pairs",median chunks/doc,max chunks/doc,docs needing a summary
domain,,,,,,,
earth_science,0.947,1.00,555,5000,47,8650,547
biology,0.980,0.98,511,5000,47,4202,508
pony,0.874,1.00,342,5000,6,401,233
psychology,0.933,0.94,480,5000,55,2079,474



Tables saved to /content/drive/MyDrive/periscope_hackathon/q50


### Reading the results
- **The claim to test:** `rows+sum (vote)` (K + 1 calls) reaches `periscope` (2K calls) while beating `rows` and `embed`.
  In Table 4, a 95% interval that contains 0 means the difference is within noise at this number of queries.
- **Context only, not comparable:** the paper's full-corpus Periscope scores NDCG@10 .688 (biology) and .379 (pony) over all ~500–600
  documents; here every arm re-ranks the same top 100, so compare arms with each other, not with the paper.
- With `N_QUERIES = 10` the tables check the pipeline, not the hypothesis. Use 50 for conclusions.